# Librerias

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler


# Lectura de datos

In [2]:
df_train = pd.read_csv("../data/raw/df_train.csv")
df_test = pd.read_csv("../data/raw/df_test.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_28296\3809680416.py:1: DtypeWarning: Columns (12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("../data/raw/df_train.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_28296\3809680416.py:2: DtypeWarning: Columns (12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv("../data/raw/df_test.csv")


In [3]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2642706 entries, 0 to 2642705
Data columns (total 17 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   date          object 
 2   store_nbr     int64  
 3   family        object 
 4   sales         float64
 5   onpromotion   int64  
 6   city          object 
 7   state         object 
 8   type_x        object 
 9   cluster       int64  
 10  transactions  float64
 11  dcoilwtico    float64
 12  type_y        object 
 13  locale        object 
 14  locale_name   object 
 15  description   object 
 16  transferred   object 
dtypes: float64(3), int64(4), object(10)
memory usage: 342.8+ MB


In [4]:
df_train.rename(columns={'type_x': 'type_store', 'type_y': 'type_holiday', 'dcoilwtico': 'oil_price'}, inplace=True)
df_test.rename(columns={'type_x': 'type_store', 'type_y': 'type_holiday', 'dcoilwtico': 'oil_price'}, inplace=True)

In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2642706 entries, 0 to 2642705
Data columns (total 17 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   date          object 
 2   store_nbr     int64  
 3   family        object 
 4   sales         float64
 5   onpromotion   int64  
 6   city          object 
 7   state         object 
 8   type_store    object 
 9   cluster       int64  
 10  transactions  float64
 11  oil_price     float64
 12  type_holiday  object 
 13  locale        object 
 14  locale_name   object 
 15  description   object 
 16  transferred   object 
dtypes: float64(3), int64(4), object(10)
memory usage: 342.8+ MB


In [6]:
#Comprobación de que están ordenadas.
start_tr = df_train['date'].head().tolist()
end_tr = df_train['date'].tail().tolist()
start_tt = df_test['date'].head().tolist()
end_tt = df_test['date'].tail().tolist()

print(f'TRAIN | start: {start_tr} ; end: {end_tr}')
print(f'TEST | start: {start_tt} ; end: {end_tt}')

TRAIN | start: ['2013-01-01', '2013-01-01', '2013-01-01', '2013-01-01', '2013-01-01'] ; end: ['2016-12-31', '2016-12-31', '2016-12-31', '2016-12-31', '2016-12-31']
TEST | start: ['2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01'] ; end: ['2017-08-15', '2017-08-15', '2017-08-15', '2017-08-15', '2017-08-15']


# Selección de Características

In [7]:
# Todas lo que escrito en la lista es lo que nos eliminaremos al final, una vez hayamos hecho las transformaciones, el feature engineering, etc.
cols_drop = ['id','store_nbr','date','city','locale','locale_name','state','description','transferred','type_holiday']

## Gestión de tipos

In [8]:
df_train['date'] = pd.to_datetime(df_train['date'])
df_test['date'] = pd.to_datetime(df_test['date'])

In [9]:
df_train['cluster']=df_train['cluster'].astype('object')
df_test['cluster']=df_test['cluster'].astype('object')

## Feature Engineering

In [10]:
def extract_date_features(df):
    """ Realiza ingeniería de características (feature engineering) sobre una columna de fechas.
    
    Extrae componentes temporales básicos y crea variables binarias de contexto 
    específicas para el análisis de ventas (fines de semana, estacionalidad de diciembre, 
    días de pago y eventos sísmicos).
    
    Args:
        df (pd.DataFrame): DataFrame que debe contener obligatoriamente una 
                           columna llamada 'date' en formato datetime.
                           
    Returns:
        pd.DataFrame: El DataFrame original con las siguientes columnas adicionales:
            - year (int): Año de la observación.
            - month (int): Mes del año (1-12).
            - day (int): Día del mes (1-31).
            - day_week (int): Día de la semana (0=Lunes, 6=Domingo).
            - is_weekend (int): 1 si es Sábado o Domingo, 0 en caso contrario.
            - is_christmas (int): 1 si el mes es Diciembre, 0 en caso contrario.
            - es_pago (int): 1 si es día 15 o último día del mes, 0 en caso contrario.
            - terremoto (int): 1 si la fecha es igual o posterior al 16 de abril de 2016."""

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_week'] = df['date'].dt.dayofweek # 0=Lunes, 6=Domingo
    
    # is_weekend: Sábado (5) y Domingo (6)
    # Vimos que la media subía de 304 a 427, ¡es clave!
    df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
    
    # is_christmas (Diciembre):
    # Según tu gráfico image_b30700.png, diciembre es el pico máximo
    df['is_christmas'] = (df['date'].dt.month == 12).astype(int)
    
    # Día de pago (Quincenas)
    df['es_pago'] = ((df['date'].dt.day == 15) | df['date'].dt.is_month_end).astype(int)
    
    # Terremoto (Variable de choque)
    df['terremoto'] = (df['date'] >= '2016-04-16').astype(int)

    #Creamos variable is_workday
    laboral = ['Work Day']

    #Es laboral (1) si el tipo es 'Work Day' O si el festivo fue transferido (transferred == True)
    df['is_workday'] = (df['type_holiday'].isin(laboral) | (df['transferred'] == True)).astype(int)
   
    return df

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)

df_train.shape, df_test.shape

((2642706, 26), (411642, 26))

# Gestión de variables categóricos

In [11]:
categoric_cols = df_train.select_dtypes(include='object').columns
categoric_cols

Index(['family', 'city', 'state', 'type_store', 'cluster', 'type_holiday',
       'locale', 'locale_name', 'description', 'transferred'],
      dtype='object')

In [12]:
# Vamos a agrupar en family en tres tipos de categorias: Alimentos, despensa y otros. Para reducir la cardinalidad

#Definición del diccionario de mapeo (basado en las 33 categorías originales)
family_mapping = {
    # ALIMENTOS_FRESCOS
    'MEATS': 'ALIMENTOS_FRESCOS', 'POULTRY': 'ALIMENTOS_FRESCOS', 
    'SEAFOOD': 'ALIMENTOS_FRESCOS', 'PRODUCE': 'ALIMENTOS_FRESCOS', 
    'DAIRY': 'ALIMENTOS_FRESCOS', 'DELI': 'ALIMENTOS_FRESCOS', 'EGGS': 'ALIMENTOS_FRESCOS',

    #ABARROTES_DESPENSA
    'GROCERY I': 'ABARROTES_DESPENSA', 'GROCERY II': 'ABARROTES_DESPENSA', 
    'BEVERAGES': 'ABARROTES_DESPENSA', 'BREAD/BAKERY': 'ABARROTES_DESPENSA', 
    'FROZEN FOODS': 'ABARROTES_DESPENSA', 'PREPARED FOODS': 'ABARROTES_DESPENSA'
}

#Aplicar la transformación de forma permanente
#Usamos .fillna('OTROS') para agrupar las 20 categorías restantes[cite: 1]
df_train['family'] = df_train['family'].map(family_mapping).fillna('OTROS')
df_test['family'] = df_test['family'].map(family_mapping).fillna('OTROS')

#Verificación de la reducción de cardinalidad[cite: 1]
print(f"Nuevas categorías únicas en family: {df_train['family'].unique()}")

Nuevas categorías únicas en family: ['OTROS' 'ABARROTES_DESPENSA' 'ALIMENTOS_FRESCOS']


In [13]:


# 1. Definimos qué columnas van a cada transformador
cols_ohe = ['family', 'type_store']
# Mantenemos store_nbr para no perder la capacidad predictiva de cada tienda
cols_target = ['cluster'] 
# Columna para el StandardScaler (ya debe estar interpolada sin nulos)
cols_scale = ['oil_price'] 

# 2. Creamos el ColumnTransformer actualizado
# 'remainder=passthrough' permite que 'transactions' (ya transformada con YJ) 
# y las binarias pasen sin cambios.
preprocessor = ColumnTransformer(
    transformers=[
        # One-Hot Encoding para categorías de baja cardinalidad
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cols_ohe),
        
        # Target Encoding para variables de alta cardinalidad
        ('cat_target', TargetEncoder(smoothing=10), cols_target),
        
        # Estandarización para el precio del petróleo
        ('num_scale', StandardScaler(), cols_scale)
    ],
    remainder='passthrough'
)

# 3. Creamos la Pipeline final
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# 4. Ajustar y transformar los datos
# IMPORTANTE: Asegúrate de haber corrido la interpolación de oil_price antes de esto
X_train_encoded = model_pipeline.fit_transform(df_train.drop(columns=['sales']), df_train['sales'])

# Para el test, usamos los parámetros (media y desviación) aprendidos en el fit de train[cite: 1]
X_test_encoded = model_pipeline.transform(df_test)

# Missing Values

In [14]:
(df_train.isnull().sum() / df_train.shape[0])*100

id               0.000000
date             0.000000
store_nbr        0.000000
family           0.000000
sales            0.000000
onpromotion      0.000000
city             0.000000
state            0.000000
type_store       0.000000
cluster          0.000000
transactions     9.224295
oil_price       31.018206
type_holiday    83.007417
locale          83.007417
locale_name     83.007417
description     83.007417
transferred     83.007417
year             0.000000
month            0.000000
day              0.000000
day_week         0.000000
is_weekend       0.000000
is_christmas     0.000000
es_pago          0.000000
terremoto        0.000000
is_workday       0.000000
dtype: float64

In [15]:
def impute_transactions(df):
    """
    Realiza la imputación de valores nulos y la transformación de potencia 
    (Yeo-Johnson) para la columna 'transactions'.
    
    La imputación sigue una lógica de negocio: si las ventas son 0, se asume que 
    la tienda estaba cerrada y se imputa 0 en transacciones. En caso contrario, 
    se utiliza la mediana del conjunto de entrenamiento para evitar el sesgo 
    de los outliers identificados en el EDA.
    
    Args:
        df_train (pd.DataFrame): Conjunto de datos de entrenamiento que contiene 
                                 las columnas 'transactions' y 'sales'.
        df_test (pd.DataFrame): Conjunto de datos de prueba/validación.
                                 
    Returns:
        tuple: (df_train, df_test) con la columna 'transactions' procesada.
    """
    
    # Calculamos la mediana del set de entrenamiento para evitar data leakage
    median_transactions = df['transactions'].median()
    
    # Caso A: Si sales es 0 y transactions es NaN -> Imputamos 0
    df.loc[(df['transactions'].isna()) & (df['sales'] == 0), 'transactions'] = 0
    
    # Caso B: Resto de NaN (tiendas abiertas sin registro) -> Imputamos la mediana
    df['transactions'] = df['transactions'].fillna(median_transactions)
    
    return df

# Aplicamos la imputación a train y test
df_train = impute_transactions(df_train)
df_test = impute_transactions(df_test)

In [16]:
def impute_oil(df):
    """
    Imputa los valores nulos del precio del petróleo usando interpolación lineal,
    ya que los nulos corresponden a fines de semana y festivos donde no hay cotización.
    """
    # 1. Aseguramos que los datos estén ordenados por fecha (muy importante para interpolar)
    df = df.sort_values(by=['store_nbr', 'date'])

#   Interpolación lineal para rellenar los huecos del fin de semana
    df['oil_price'] = df['oil_price'].interpolate(method='linear')

    #ffill() y bfill() por si el primer o último registro del dataset es nulo
    df['oil_price'] = df['oil_price'].ffill().bfill()

    return df

#Aplicamos la función a Train y Test
df_train = impute_oil(df_train)
df_test = impute_oil(df_test)

#Verificación
print(f"Nulos en oil_price (Train): {df_train['oil_price'].isna().sum()}")

Nulos en oil_price (Train): 0


# Outliers

In [17]:
print("--- Resumen de Outliers ---")
for col in df_train.select_dtypes(exclude='object'):
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)]
    perc = (len(outliers) / len(df_train)) * 100
    print(f"{col}: {len(outliers)} registros ({perc:.2f}%)")

--- Resumen de Outliers ---
id: 0 registros (0.00%)
date: 0 registros (0.00%)
store_nbr: 0 registros (0.00%)
sales: 395096 registros (14.95%)
onpromotion: 439508 registros (16.63%)
transactions: 141009 registros (5.34%)
oil_price: 0 registros (0.00%)
year: 0 registros (0.00%)
month: 0 registros (0.00%)
day: 0 registros (0.00%)
day_week: 0 registros (0.00%)
is_weekend: 0 registros (0.00%)
is_christmas: 222750 registros (8.43%)
es_pago: 171072 registros (6.47%)
terremoto: 481140 registros (18.21%)
is_workday: 17820 registros (0.67%)


In [18]:
log_cols = ['onpromotion', 'sales']

# Sobreescribe las columnas originales con su versión logarítmica
df_train[log_cols] = np.log1p(df_train[log_cols])
df_test[log_cols] = np.log1p(df_test[log_cols])

In [19]:
df_train.describe()

,id,date,store_nbr,sales,onpromotion,transactions,oil_price,year,month,day,day_week,is_weekend,is_christmas,es_pago,terremoto,is_workday
count,2.642706e+06,2642706,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06,2.642706e+06
mean,1.302440e+06,2015-01-02 21:50:51.382333696,2.750000e+01,2.829295e+00,2.893839e-01,1.548823e+03,7.069656e+01,2.014506e+03,6.521241e+00,1.573095e+01,3.008092e+00,2.872556e-01,8.428860e-02,6.473365e-02,1.820634e-01,6.743088e-03
min,0.000000e+00,2013-01-01 00:00:00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.619000e+01,2.013000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.517662e+05,2014-01-02 00:00:00,1.400000e+01,0.000000e+00,0.000000e+00,9.180000e+02,4.621000e+01,2.014000e+03,4.000000e+00,8.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.301750e+06,2015-01-03 00:00:00,2.750000e+01,2.302585e+00,0.000000e+00,1.328000e+03,5.980045e+01,2.015000e+03,7.000000e+00,1.600000e+01,3.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.955299e+06,2016-01-06 00:00:00,4.100000e+01,5.225747e+00,0.000000e+00,1.984000e+03,9.699000e+01,2.016000e+03,9.000000e+00,2.300000e+01,5.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2.596373e+06,2016-12-31 00:00:00,5.400000e+01,1.173381e+01,6.609349e+00,8.359000e+03,1.106200e+02,2.016000e+03,1.200000e+01,3.100000e+01,6.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
std,7.502727e+05,NaN,1.558579e+01,2.703491e+00,7.927439e-01,1.047843e+03,2.644159e+01,1.120125e+00,3.430073e+00,8.804986e+00,1.997960e+00,4.524819e-01,2.778202e-01,2.460553e-01,3.858968e-01,8.183900e-02


In [20]:
print("--- Resumen de Outliers ---")
for col in df_train.select_dtypes(exclude='object'):
    Q1 = df_train[col].quantile(0.25)
    Q3 = df_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)]
    perc = (len(outliers) / len(df_train)) * 100
    print(f"{col}: {len(outliers)} registros ({perc:.2f}%)")

--- Resumen de Outliers ---
id: 0 registros (0.00%)
date: 0 registros (0.00%)
store_nbr: 0 registros (0.00%)
sales: 0 registros (0.00%)
onpromotion: 439508 registros (16.63%)
transactions: 141009 registros (5.34%)
oil_price: 0 registros (0.00%)
year: 0 registros (0.00%)
month: 0 registros (0.00%)
day: 0 registros (0.00%)
day_week: 0 registros (0.00%)
is_weekend: 0 registros (0.00%)
is_christmas: 222750 registros (8.43%)
es_pago: 171072 registros (6.47%)
terremoto: 481140 registros (18.21%)
is_workday: 17820 registros (0.67%)


# Normalización/ Estandarización

In [21]:
pt = PowerTransformer(method="yeo-johnson")

# Es importante ajustar (fit) solo con los datos de entrenamiento
df_train['transactions'] = pt.fit_transform(df_train[['transactions']])

# Y transformar (transform) los datos de test con los parámetros aprendidos
df_test['transactions'] = pt.transform(df_test[['transactions']])

print("Nulos en transactions tras imputación:", df_train['transactions'].isna().sum())

Nulos en transactions tras imputación: 0


In [22]:
# Aplicamos la eliminación y guardamos el resultado
df_train = df_train.drop(columns=cols_drop)
df_test = df_test.drop(columns=cols_drop)

# Verificación rápida del "equipo final" de columnas
print(f"Columnas finales en Train ({len(df_train.columns)}): {df_train.columns.tolist()}")

Columnas finales en Train (16): ['family', 'sales', 'onpromotion', 'type_store', 'cluster', 'transactions', 'oil_price', 'year', 'month', 'day', 'day_week', 'is_weekend', 'is_christmas', 'es_pago', 'terremoto', 'is_workday']


In [26]:
df_train.head()


,family,sales,onpromotion,type_store,cluster,transactions,oil_price,year,month,day,day_week,is_weekend,is_christmas,es_pago,terremoto,is_workday
0,OTROS,0.0,0.0,D,13,-2.245644,93.14,2013,1,1,1,0,0,0,0,0
1171,ALIMENTOS_FRESCOS,0.0,0.0,D,13,-2.245644,93.14,2013,1,1,1,0,0,0,0,0
1172,OTROS,0.0,0.0,D,13,-2.245644,93.14,2013,1,1,1,0,0,0,0,0
1173,OTROS,0.0,0.0,D,13,-2.245644,93.14,2013,1,1,1,0,0,0,0,0
1174,OTROS,0.0,0.0,D,13,-2.245644,93.14,2013,1,1,1,0,0,0,0,0


# Exportación de datos

In [27]:
y_train=df_train['sales']
y_test=df_test['sales']

In [29]:

import os

# 1. Recuperar los nombres de las columnas del preprocesador
# Esto evita que tu CSV tenga columnas llamadas "0, 1, 2..."
column_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 2. Convertir a DataFrame con sus nombres reales
X_train_final_df = pd.DataFrame(X_train_encoded, columns=column_names)
X_test_final_df = pd.DataFrame(X_test_encoded, columns=column_names)

# 3. Resetear índices de los targets (como tú hiciste, ¡es fundamental!)
y_train_res = y_train.reset_index(drop=True)
y_test_res = y_test.reset_index(drop=True)

# 4. Concatenar (en el test no concatenamos 'y' porque es lo que vamos a predecir)
df_train_final = pd.concat([X_train_final_df, y_train_res], axis=1)
# El test se guarda solo (sin la columna sales, que es el objetivo)
df_test_final = X_test_final_df 

# 5. Exportar a la ruta relativa
output_path = '../data/clean'
if not os.path.exists(output_path):
    os.makedirs(output_path)

df_train_final.to_csv(f'{output_path}/train_clean.csv', index=False)
df_test_final.to_csv(f'{output_path}/test_clean.csv', index=False)

print("¡Archivos listos para el modelo en ../data/clean/!")

¡Archivos listos para el modelo en ../data/clean/!
